In [2]:
%matplotlib inline
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from game import Game
from player import Player

In [1]:
from MyFirstPlayer import MyFirstPlayer
from MySecondPlayer import MySecondPlayer
from Kondys_Dabrowski import Kondys_Dabrowski
from bot import KK

In [3]:
### Generate cards from 9 to 14 (ace) for all colors/symbols (0, 1, 2, 3)
def getDeck():
    return [(number, color) for color in range(4) for number in range(9, 15)]
    
print(getDeck())

[(9, 0), (10, 0), (11, 0), (12, 0), (13, 0), (14, 0), (9, 1), (10, 1), (11, 1), (12, 1), (13, 1), (14, 1), (9, 2), (10, 2), (11, 2), (12, 2), (13, 2), (14, 2), (9, 3), (10, 3), (11, 3), (12, 3), (13, 3), (14, 3)]


In [4]:
### Shuffle the cards randomly. Each player gets 9 cards
### (so one player cannot be certain which cards the other player has)

def getShuffled(deck):
    D = set(deck)
    A = set(random.sample(deck, 8))
    B = set(random.sample(list(D - A), 8))
    C = D - A - B
    if len(A.intersection(B)) > 0: print("Shuffle error 1")
    if len(A.intersection(B)) > 0: print("Shuffle error 2")
    if len(A.intersection(C)) > 0: print("Shuffle error 3") 
    DS = A | B | C
    if not DS == D: print("Shuffle error 4")  
    return list(A), list(B), list(C)

p1, p2, notUsed, = getShuffled(getDeck())
print(p1)
print(p2)


[(14, 1), (11, 3), (10, 1), (9, 2), (13, 0), (13, 3), (14, 2), (10, 2)]
[(11, 1), (9, 3), (12, 0), (14, 0), (11, 2), (12, 2), (10, 3), (13, 2)]


In [5]:
class DrawPlayer(Player):
    
    ### player's random strategy
    def putCard(self, declared_card):
        return "draw"
    
    ### randomly decides whether to check or not
    def checkCard(self, opponent_declaration):
        return np.random.choice([False, False])
    
class SimplePlayer(Player):
    
    ### player's simple strategy
    def putCard(self, declared_card):
        
        ### check if must draw
        if len(self.cards) == 1 and declared_card is not None and self.cards[0][0] < declared_card[0]:
            return "draw"
        
        card = min(self.cards, key=lambda x: x[0])
        declaration = (card[0], card[1])
        if declared_card is not None:
            min_val = declared_card[0]
            if card[0] < min_val: declaration = (min(min_val + 1, 14), declaration[1])
        return card, declaration
    
    def checkCard(self, opponent_declaration):
        if opponent_declaration in self.cards: return True
        return np.random.choice([True, False], p=[0.3, 0.7])
class RandomPlayer(Player):
    
    ### player's random strategy
    def putCard(self, declared_card):
        
        ### check if must draw
        if len(self.cards) == 1 and declared_card is not None and self.cards[0][0] < declared_card[0]:
            return "draw"
        
        ### player randomly decides which card put on the table
        card = random.choice(self.cards)
        declaration = card
        
        ### player randomly decides whether to cheat or not
        cheat = np.random.choice([True, False])
       
        ### if (s)he decides to cheat, (s)he randomly declares the card.
        if cheat:
            declaration = random.choice(self.cards)             
            
        ### Yet, declared card should be no worse than a card on the top of the pile . 
        if declared_card is not None and declaration[0] < declared_card[0]:
            declaration = (min(declared_card[0]+1,14), declaration[1])

        ### return the decision (true card) and declaration (player's declaration)
        return card, declaration
    
    ### randomly decides whether to check or not
    def checkCard(self, opponent_declaration):
        return np.random.choice([True, False])
    

Analyze few moves...

In [6]:
### Perform a full game 100 times
STAT_NAMES = ["Wins", "Draws", "Moves", "Cards", "Pile Size", "Checks", "Draw Decisions", "Cheats", "Errors", "Total errors"]
ANALYZE_STATS = [0, 1, 2, 3, 5, 6, 7, 8]

def printResults(results):
    print("Wins:")
    print(results[0])
    print("Draws:")
    print(results[1])
    print("Moves:")
    print(results[2])
    print("Cards:")
    print(results[3])
    print("Pile size:")
    print(results[4])
    print("Checks:")
    print(results[5])
    print("Draw decisions:")
    print(results[6])
    print("Cheats:")
    print(results[7])
    print("Errors:")
    print(results[8])
    print("Total errors:")
    print(results[9])

def comparePlayers(player1_class, player2_class):
    stats_wins = [0, 0]
    stats_draws = [0, 0]
    stats_moves = [0, 0]
    stats_cards = [0, 0]
    stats_pile_size = 0
    stats_checks = [0, 0]
    stats_draw_decisions = [0, 0]
    stats_cheats = [0, 0]
    stats_errors = [0, 0]
    
    repeats = 1000
    errors = 0
    draws = 0

    for t in range(repeats):
        player1 = player1_class("")
        player2 = player2_class("")
        game = Game([player1, player2], log = False)
    
        error = False
        draw = False
        
        while True:
            valid, player = game.takeTurn(log = False)
            if game.moves[0] > 100 or game.moves[1] > 100:
                draws += 1
                stats_draws[0] += 1
                stats_draws[1] += 1
                if (game.player_cards[0] < game.player_cards[1]):
                    stats_wins[0] += 1
                if (game.player_cards[0] > game.player_cards[1]):
                    stats_wins[1] += 1
                    
                    
                draw=True
                # print("DRAW")
                break
            if not valid:
                error = True
                stats_errors[player] += 1
                errors += 1
                break
            if game.isFinished(log = False):
                stats_wins[player] += 1
                break
            
        stats_pile_size += len(game.pile)
        if error: continue
        #if draw: continue
       
        for j in range(2):
            stats_moves[j] += game.moves[j]
            stats_cheats[j] += game.cheats[j]
            stats_checks[j] += game.checks[j]
            stats_draw_decisions[j] += game.draw_decisions[j]
            stats_cards[j] += len(game.player_cards[j])

    div = repeats - errors
    if div > 0:
            
        stats_pile_size /= div          
        for j in range(2):
            stats_moves[j] /= div
            stats_cheats[j] /= div
            stats_checks[j] /= div
            stats_draw_decisions[j] /= div
            stats_cards[j] /= div
            
    return [stats_wins, stats_draws, stats_moves, stats_cards, stats_pile_size, stats_checks, 
            stats_draw_decisions, stats_cheats, stats_errors, errors, draws]  


# COMPARE

In [12]:
strategy = [["RandomPlayer", "RANDOM", RandomPlayer],
            ["Kacper", "Jakub", Kondys_Dabrowski],
            ["Kot1", "Kot1", MyFirstPlayer],
            ["SimplePlayer", "SimplePlayer", SimplePlayer],
            ["Kot2","Kot2", MySecondPlayer],
            ["dsad","dsadsa", KK]
           ]

In [8]:
#%pdb on
full_results = [[None for i in range(len(strategy))] for i in range(len(strategy))]

for A in range(len(strategy)):
    print("==== " + str(A), strategy[A][0])
    for B in range(A+1,len(strategy)):
        print(B, strategy[B][0])
        results = comparePlayers(strategy[A][2], strategy[B][2])
        full_results[A][B] = results
        


==== 0 RandomPlayer
1 Kacper
2 Kot1
3 SimplePlayer
4 Kot2
5 dsad
==== 1 Kacper
2 Kot1
3 SimplePlayer
4 Kot2
5 dsad
==== 2 Kot1
3 SimplePlayer
4 Kot2
5 dsad
==== 3 SimplePlayer
4 Kot2
5 dsad
==== 4 Kot2
5 dsad
==== 5 dsad


In [9]:
full_results

[[None,
  [[3, 997],
   [0, 0],
   [11.265, 11.745],
   [15.07, 0.033],
   0.897,
   [5.938, 5.559],
   [0.001, 0.005],
   [6.483, 0.366],
   [0, 0],
   0,
   0],
  [[50, 950],
   [0, 0],
   [15.136, 15.588],
   [12.978, 0.288],
   2.734,
   [7.592, 3.606],
   [0.021, 0.493],
   [8.599, 2.449],
   [0, 0],
   0,
   0],
  [[98, 902],
   [0, 0],
   [15.174, 15.589],
   [12.858, 1.305],
   1.837,
   [7.77, 5.915],
   [0.036, 0.022],
   [8.93, 4.287],
   [0, 0],
   0,
   0],
  [[115, 885],
   [0, 0],
   [15.491, 15.863],
   [12.407, 1.231],
   2.362,
   [7.9, 5.835],
   [0.088, 0.046],
   [9.028, 4.45],
   [0, 0],
   0,
   0],
  [[35, 965],
   [0, 0],
   [21.411, 21.902],
   [14.047, 0.303],
   1.65,
   [10.596, 8.7],
   [0.023, 0.633],
   [12.273, 4.343],
   [0, 0],
   0,
   0]],
 [None,
  None,
  [[653, 347],
   [20, 20],
   [28.902, 28.696],
   [3.818, 8.384],
   3.798,
   [13.435, 5.711],
   [0.002, 0.172],
   [0.397, 5.248],
   [0, 0],
   0,
   20],
  [[933, 67],
   [0, 0],
   [9.362, 

Simple stats

In [10]:
def printMatrix(full_results, stat):
    print(STAT_NAMES[stat])
    S = " "
    for s in strategy: S += (str(s[1]) + " " )
    print(S)
    for A in range(len(strategy)):
        print(A)
        S = str(strategy[A][1]) + " "
        for B in range(len(strategy)):
            if A == B: S += "- "
            elif A < B:
                S += str(full_results[A][B][stat][0]) + " "
            else:
                S += str(full_results[B][A][stat][1]) + " "
        print(S)
    
for a in ANALYZE_STATS:
    printMatrix(full_results, a)



Wins
 RANDOM Jakub Kot1 SimplePlayer Kot2 dsadsa 
0
RANDOM - 3 50 98 115 35 
1
Jakub 997 - 653 933 838 926 
2
Kot1 950 347 - 691 220 836 
3
SimplePlayer 902 67 309 - 347 717 
4
Kot2 885 162 780 653 - 805 
5
dsadsa 965 74 164 283 195 - 
Draws
 RANDOM Jakub Kot1 SimplePlayer Kot2 dsadsa 
0
RANDOM - 0 0 0 0 0 
1
Jakub 0 - 20 0 0 0 
2
Kot1 0 20 - 0 0 0 
3
SimplePlayer 0 0 0 - 0 0 
4
Kot2 0 0 0 0 - 0 
5
dsadsa 0 0 0 0 0 - 
Moves
 RANDOM Jakub Kot1 SimplePlayer Kot2 dsadsa 
0
RANDOM - 11.265 15.136 15.174 15.491 21.411 
1
Jakub 11.745 - 28.902 9.362 9.347 19.607 
2
Kot1 15.588 28.696 - 14.112 12.09 17.462 
3
SimplePlayer 15.589 8.947 13.912 - 12.292 21.087 
4
Kot2 15.863 9.008 12.39 12.47 - 14.786 
5
dsadsa 21.902 19.204 17.097 20.872 14.462 - 
Cards
 RANDOM Jakub Kot1 SimplePlayer Kot2 dsadsa 
0
RANDOM - 15.07 12.978 12.858 12.407 14.047 
1
Jakub 0.033 - 3.818 0.85 2.026 0.912 
2
Kot1 0.288 8.384 - 2.176 4.683 1.04 
3
SimplePlayer 1.305 13.071 8.155 - 7.884 3.559 
4
Kot2 1.231 11.496 1.642 

# to już dodałem od siebie, tego nie ma w komparatorze!!!!
tu jest fragment kodu który zrobi ranking, na podstawie czysto liczby zwycięstw

In [11]:
strategy = [
            # ["SecondPlayerOptim2", "SecondPlayerOptim2", MySecondPlayerOptim],
            ["Kot_First", "Kot_First", MyFirstPlayer],
            ["RandomPlayer", "RANDOM", RandomPlayer],
            # # # # # # # # ["Nazwisko", "Nazwisko", Nazwisko], # ten jest zaimportowany, ale jest pusty praktycznie, jak drawPlayer
            ["SimplePlayer", "SimplePlayer", SimplePlayer],
            ["NoCheck", "NoCheck", SimpleNoCheck],
            ["AlwaysCheck", "AlwaysCheck", SimpleAlwaysCheck],
            ["HonestPlayer", "HonestPlayer", HonestPlayer],
            # # # # # # # # ["DrawPlayer", "DrawPlayer", DrawPlayer], # no dobra, on jest zbyt chujowy by go brać pod uwagę ngl
            ["Kot_Second", "Kot_Second", MySecondPlayer],
            ["Kot_Third", "Kot_Third", MyThirdPlayer],
            # ["FirstCardCounter", "FirstCardCounter", FirstCardCounter],
            # ["SecondCardCounter", "SecondCardCounter", SecondCardCounter],
            # ["Mr.Ciekawostka", "Mr.Ciekawostka", MySecondPlayer],
            # ["SecondPlayerOptim", "SecondPlayerOptim", MySecondPlayerOptim],
            ### broken ["Kondys_Dabrowski", "Kondys_Dabrowski", Kondys_Dabrowski],
           ]

#%pdb on
full_results = [[None for i in range(len(strategy))] for i in range(len(strategy))]

for A in range(len(strategy)):
    # print("==== " + str(A), strategy[A][0])
    for B in range(A+1,len(strategy)):
        # print(B, strategy[B][0])
        results = comparePlayers(strategy[A][2], strategy[B][2])
        full_results[A][B] = results


def generate_ranking(full_results, strategy):
    n = len(strategy)
    total_wins = [0 for _ in range(n)]

    for i in range(n):
        for j in range(n):
            if i < j and full_results[i][j] is not None:
                wins_i, wins_j = full_results[i][j][0]
                total_wins[i] += wins_i
                total_wins[j] += wins_j

    # Sparuj nazwy z wynikami
    ranking = list(zip([s[0] for s in strategy], total_wins))

    # Posortuj malejąco
    ranking.sort(key=lambda x: x[1], reverse=True)

    maxxx = (len(strategy) -1) * 1000

    print("RANKING GRACZY (według liczby wygranych):")
    for idx, (name, wins) in enumerate(ranking, 1):
        print(f"{idx}. {name} - {wins} / {maxxx} wygranych")

    return ranking

ranking = generate_ranking(full_results, strategy)

NameError: name 'SimpleNoCheck' is not defined

In [13]:
import pandas as pd
import numpy as np

def create_win_matrix(full_results, strategy):
    names = [s[0] for s in strategy]
    n = len(names)
    matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i == j:
                matrix[i][j] = np.nan  # Brak pojedynków z samym sobą
            elif i < j and full_results[i][j] is not None:
                wins_i, wins_j = full_results[i][j][0]
                matrix[i][j] = wins_i
                matrix[j][i] = wins_j

    df = pd.DataFrame(matrix, index=names, columns=names)
    print("MACIERZ WYGRANYCH:")
    print(df.round(1))

    return df


df = create_win_matrix(full_results, strategy)

MACIERZ WYGRANYCH:
              RandomPlayer  Kacper   Kot1  SimplePlayer   Kot2   dsad
RandomPlayer           NaN     3.0   50.0          98.0  115.0   35.0
Kacper               997.0     NaN  653.0         933.0  838.0  926.0
Kot1                 950.0   347.0    NaN         691.0  220.0  836.0
SimplePlayer         902.0    67.0  309.0           NaN  347.0  717.0
Kot2                 885.0   162.0  780.0         653.0    NaN  805.0
dsad                 965.0    74.0  164.0         283.0  195.0    NaN
